# Hoja de trabajo 2 — Transformers y mecanismos de atención
**CC3092 Deep Learning y Sistemas Inteligentes** · Fernando Andree Hernández Martínez (23645)

Contenido del notebook:
1. Configuración y semillas
2. Ejercicio a mano de *scaled dot-product attention* y su verificación numérica
3. Verificaciones contra `F.scaled_dot_product_attention`, `nn.MultiheadAttention` y máscaras causales
4. Escalamiento por $\sqrt{d_k}$: varianza de $q\cdot k$, saturación de la softmax y gradientes
5. Costo computacional de la self-attention respecto a $n$
6. Mapas de atención en BERT y GPT-2 (patrones por cabeza)
7. Correferencia de «it» (*tired* vs *wide*)
8. GPT-2: matriz triangular inferior y *attention sinks*
9. Entropía de la atención por capa
10. (Opcional) bertviz

In [ ]:
# Configuración: semillas fijas para reproducibilidad y rutas del proyecto
import sys, math, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)

from attention import (scaled_dot_product_attention, causal_mask, multi_head_attention,
                       get_attentions, attention_entropy, head_patterns, plot_heads)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
print("torch", torch.__version__)

## 2. Ejercicio a mano

Tres tokens con $d_k = d_v = 2$ y, para que las cuentas sean manejables, $W_Q=W_K=W_V=I$, de modo que $Q=K=V=X$:

$$X=\begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$$

**Paso 1 — puntajes** $S = QK^\top$:
$S=\begin{pmatrix}1&0&1\\0&1&1\\1&1&2\end{pmatrix}$

**Paso 2 — escalar** por $\sqrt{d_k}=\sqrt2\approx1.4142$:
$S/\sqrt2=\begin{pmatrix}0.7071&0&0.7071\\0&0.7071&0.7071\\0.7071&0.7071&1.4142\end{pmatrix}$

**Paso 3 — softmax por fila.** Fila 1: $e^{0.7071}=2.0281$, $e^0=1$ → suma $5.0562$ → $(0.4011,\,0.1978,\,0.4011)$.
Fila 2 es la permutación de la fila 1: $(0.1978,\,0.4011,\,0.4011)$.
Fila 3: $e^{0.7071}=2.0281$, $e^{1.4142}=4.1133$ → suma $8.1695$ → $(0.2483,\,0.2483,\,0.5035)$.

**Paso 4 — salida** $Z = AV$:
- $z_1 = 0.4011(1,0)+0.1978(0,1)+0.4011(1,1) = (0.8022,\,0.5989)$
- $z_2 = (0.5989,\,0.8022)$
- $z_3 = 0.2483(1,0)+0.2483(0,1)+0.5035(1,1) = (0.7517,\,0.7517)$

Interpretación: el token 3 ($[1,1]$) es el más parecido a todos, por eso recibe más peso en todas las filas; cada salida es un promedio ponderado de los valores.

In [ ]:
# Verificación numérica del ejercicio a mano
X = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
S = X @ X.T
S_scaled = S / math.sqrt(2)
A_hand = torch.softmax(S_scaled, dim=-1)
Z_hand = A_hand @ X
print("S =\n", S, "\nS/sqrt(2) =\n", S_scaled, "\nA =\n", A_hand, "\nZ = A V =\n", Z_hand)

Z_ref = F.scaled_dot_product_attention(X[None], X[None], X[None])[0]
A_expected = torch.tensor([[0.4011, 0.1978, 0.4011], [0.1978, 0.4011, 0.4011], [0.2483, 0.2483, 0.5035]])
Z_expected = torch.tensor([[0.8022, 0.5989], [0.5989, 0.8022], [0.7517, 0.7517]])
print("A coincide con lo calculado a mano:", torch.allclose(A_hand, A_expected, atol=1e-4))
print("Z coincide con lo calculado a mano:", torch.allclose(Z_hand, Z_expected, atol=1e-4))
print("Z coincide con F.scaled_dot_product_attention:", torch.allclose(Z_hand, Z_ref, atol=1e-6))

## 3. Verificaciones de la implementación

Se compara la implementación propia (`src/attention.py`) con las de PyTorch:
1. `scaled_dot_product_attention` propia vs `F.scaled_dot_product_attention` (sin máscara y con `is_causal=True`).
2. `multi_head_attention` propia vs `nn.MultiheadAttention` copiando sus pesos (`in_proj_weight`, `out_proj.weight`).
3. Propiedades: filas de la atención suman 1, la máscara causal produce una matriz triangular inferior, y `nn.Transformer.generate_square_subsequent_mask` equivale a nuestra máscara booleana.

In [ ]:
B, h, n, d = 2, 4, 7, 16
Q, K, V = torch.randn(B, h, n, d), torch.randn(B, h, n, d), torch.randn(B, h, n, d)

out_mine, w_mine = scaled_dot_product_attention(Q, K, V)
out_torch = F.scaled_dot_product_attention(Q, K, V)
out_mine_c, w_mine_c = scaled_dot_product_attention(Q, K, V, mask=causal_mask(n))
out_torch_c = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

# nn.MultiheadAttention con los mismos pesos que nuestra implementación
d_model, heads = 32, 4
mha = nn.MultiheadAttention(d_model, heads, batch_first=True, bias=False).eval()
x = torch.randn(B, n, d_model)
W_q, W_k, W_v = mha.in_proj_weight.chunk(3)
ours, w_ours = multi_head_attention(x, W_q, W_k, W_v, mha.out_proj.weight, heads)
ref, w_ref = mha(x, x, x, need_weights=True, average_attn_weights=False)

# Máscara de PyTorch: float con -inf arriba de la diagonal y 0 en el resto
torch_mask = nn.Transformer.generate_square_subsequent_mask(n)

checks = {
    "SDPA propia == F.scaled_dot_product_attention": torch.allclose(out_mine, out_torch, atol=1e-5),
    "SDPA causal propia == F.sdpa(is_causal=True)": torch.allclose(out_mine_c, out_torch_c, atol=1e-5),
    "MHA propia == nn.MultiheadAttention (salida)": torch.allclose(ours, ref, atol=1e-5),
    "MHA propia == nn.MultiheadAttention (pesos)": torch.allclose(w_ours, w_ref, atol=1e-5),
    "Filas de atención suman 1": torch.allclose(w_mine.sum(-1), torch.ones(B, h, n)),
    "Atención causal es triangular inferior": bool((w_mine_c.triu(1) == 0).all()),
    "generate_square_subsequent_mask == ~causal_mask": torch.equal(torch.isinf(torch_mask), ~causal_mask(n)),
}
for k, v in checks.items():
    print(f"[{'OK' if v else 'FALLA'}] {k}")
print("Error máximo MHA:", (ours - ref).abs().max().item())

## 4. ¿Por qué dividir entre $\sqrt{d_k}$?

Si $q_i, k_i \sim \mathcal N(0,1)$ independientes, $q\cdot k=\sum_{i=1}^{d_k} q_ik_i$ es suma de $d_k$ términos con media 0 y varianza $E[q_i^2]E[k_i^2]=1$, así que $\operatorname{Var}(q\cdot k)=d_k$. Dividir entre $\sqrt{d_k}$ devuelve la varianza a 1. Se verifica empíricamente y se mide el efecto sobre la softmax (probabilidad máxima) y sobre la norma del gradiente de la softmax.

In [ ]:
dks = [4, 16, 64, 256, 1024]
n_tokens, trials = 32, 2000
rows = []
for dk in dks:
    q = torch.randn(trials, dk); k = torch.randn(trials, dk)
    var_raw = (q * k).sum(-1).var().item()
    var_scaled = ((q * k).sum(-1) / math.sqrt(dk)).var().item()
    res = {}
    for scaled in (False, True):
        Qs = torch.randn(200, 1, dk); Ks = torch.randn(200, n_tokens, dk)
        logits = (Qs @ Ks.transpose(-2, -1)).squeeze(1)
        if scaled:
            logits = logits / math.sqrt(dk)
        logits.requires_grad_(True)
        p = torch.softmax(logits, -1)
        # gradiente de la softmax respecto a sus logits con una señal aleatoria aguas arriba
        (p * torch.randn_like(p)).sum().backward()
        res[scaled] = (p.max(-1).values.mean().item(), logits.grad.norm(dim=-1).mean().item())
    rows.append((dk, var_raw, var_scaled, *res[False], *res[True]))

print(f"{'d_k':>5} {'Var(q·k)':>9} {'Var esc.':>9} {'max p s/esc':>11} {'|grad| s/esc':>12} {'max p esc':>10} {'|grad| esc':>10}")
for r in rows:
    print(f"{r[0]:>5} {r[1]:>9.1f} {r[2]:>9.3f} {r[3]:>11.3f} {r[4]:>12.4f} {r[5]:>10.3f} {r[6]:>10.4f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
dk_arr = np.array(dks)
axes[0].loglog(dk_arr, [r[1] for r in rows], "o-", label="sin escalar")
axes[0].loglog(dk_arr, [r[2] for r in rows], "s-", label=r"escalado $/\sqrt{d_k}$")
axes[0].loglog(dk_arr, dk_arr, "k--", lw=0.8, label="$d_k$ (teórico)")
axes[0].set(title=r"Varianza de $q\cdot k$", xlabel="$d_k$"); axes[0].legend(fontsize=8)
axes[1].semilogx(dk_arr, [r[3] for r in rows], "o-", label="sin escalar")
axes[1].semilogx(dk_arr, [r[5] for r in rows], "s-", label="escalado")
axes[1].axhline(1 / n_tokens, color="k", ls="--", lw=0.8, label="uniforme 1/n")
axes[1].set(title="Prob. máxima de la softmax (n=32)", xlabel="$d_k$"); axes[1].legend(fontsize=8)
axes[2].loglog(dk_arr, [r[4] for r in rows], "o-", label="sin escalar")
axes[2].loglog(dk_arr, [r[6] for r in rows], "s-", label="escalado")
axes[2].set(title="Norma del gradiente de la softmax", xlabel="$d_k$"); axes[2].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "escalamiento.png"); plt.show()

**Lectura:** sin escalar, la varianza crece linealmente con $d_k$; los logits se separan tanto que la softmax se vuelve casi *one-hot* (prob. máxima → 1) y su jacobiano $\operatorname{diag}(p)-pp^\top$ tiende a cero, por lo que el gradiente se desvanece. Con el escalamiento la varianza se mantiene ≈1 y la distribución y el gradiente quedan estables para cualquier $d_k$.

## 5. Costo computacional respecto a $n$

Se mide tiempo y memoria de la matriz de atención para longitudes crecientes ($d=64$, una cabeza, CPU). La memoria de la matriz $n\times n$ en float32 es $4n^2$ bytes; el tiempo debería crecer como $O(n^2 d)$.

In [ ]:
def bench(fn, reps=5):
    fn()  # calentamiento
    t = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t) / reps

d = 64
ns = [256, 512, 1024, 2048, 4096, 8192]
n_threads = torch.get_num_threads()
torch.set_num_threads(1)  # un solo hilo: el tiempo refleja el número de operaciones, no el paralelismo
t_manual, t_fused, mem_mb = [], [], []
for n_ in ns:
    q = torch.randn(1, 1, n_, d); k = torch.randn(1, 1, n_, d); v = torch.randn(1, 1, n_, d)
    t_manual.append(bench(lambda: scaled_dot_product_attention(q, k, v)))
    t_fused.append(bench(lambda: F.scaled_dot_product_attention(q, k, v)))
    mem_mb.append(4 * n_ * n_ / 2**20)  # matriz de pesos n×n en float32
torch.set_num_threads(n_threads)

slope = np.polyfit(np.log(ns[2:]), np.log(t_manual[2:]), 1)[0]
print("Pendiente log-log del tiempo (implementación manual):", round(slope, 2))
for n_, a, b, m in zip(ns, t_manual, t_fused, mem_mb):
    print(f"n={n_:>5}  manual={a*1e3:8.2f} ms  F.sdpa={b*1e3:8.2f} ms  matriz n×n={m:8.1f} MB")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].loglog(ns, np.array(t_manual) * 1e3, "o-", label="manual (materializa n×n)")
axes[0].loglog(ns, np.array(t_fused) * 1e3, "s-", label="F.scaled_dot_product_attention")
ref = np.array(ns, float) ** 2; axes[0].loglog(ns, ref / ref[-1] * t_manual[-1] * 1e3, "k--", lw=0.8, label="$O(n^2)$")
axes[0].set(title="Tiempo de self-attention", xlabel="n (tokens)", ylabel="ms"); axes[0].legend(fontsize=8)
n_big = np.array([1e3, 4e3, 16e3, 32e3, 100e3])
axes[1].loglog(ns, mem_mb, "o-", label="medido (n×n float32)")
axes[1].loglog(n_big, 4 * n_big**2 / 2**30 * 1024, "k--", lw=0.8, label="extrapolado")
axes[1].axhline(80 * 1024, color="r", ls=":", label="80 GB (GPU H100)")
axes[1].set(title="Memoria de la matriz de atención\n(1 cabeza, 1 capa)", xlabel="n (tokens)", ylabel="MB"); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "costo.png"); plt.show()
print(f"n=100,000 → {4*1e5**2/2**30:.1f} GB por cabeza y capa sólo para la matriz de atención")

**Lectura:** el tiempo crece con pendiente ≈2 en escala log-log para $n\ge1024$ (cuadrático; con $n$ pequeño domina el overhead) y la memoria de la matriz es exactamente $4n^2$ bytes: con $n=100{,}000$ serían ~37 GB *por cabeza y por capa*. La versión fusionada de PyTorch es más rápida porque no siempre materializa la matriz completa (kernels tipo FlashAttention / memory-efficient), pero el número de operaciones sigue siendo $O(n^2d)$.